In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [2]:
DATA_PATH = "../data/raw/telco_customer_churn.csv"
df = pd.read_csv(DATA_PATH)
print("Original shape:", df.shape)

Original shape: (7043, 33)


In [3]:
TARGET = "Churn Label"
y = df[TARGET].map({"No": 0,"Yes": 1})

X = df.drop(columns=[TARGET]).copy()

columns_to_drop = ["Count","Country","State","CustomerID","Churn Value","Churn Score","Churn Reason"]

X = X.drop(columns=columns_to_drop)
X["Total Charges"] = pd.to_numeric(X["Total Charges"],errors="coerce")

X.loc[(X["Tenure Months"] == 0) & (X["Total Charges"].isna()),"Total Charges"] = 0
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 25)
y shape: (7043,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

X_train: (5634, 25)
X_test : (1409, 25)


In [5]:
numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()

categorical_features = X_train.select_dtypes(include=["object", "string"]).columns.tolist()
print("Numerical features:")
print(numerical_features)
print("\nCategorical features:")
print(categorical_features)

print("\nNumerical feature count:", len(numerical_features))
print("Categorical feature count:", len(categorical_features))

Numerical features:
['Zip Code', 'Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV']

Categorical features:
['City', 'Lat Long', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']

Numerical feature count: 7
Categorical feature count: 18


In [6]:
numerical_pipeline = Pipeline(steps=[("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])

In [7]:
categorical_pipeline = Pipeline(steps=[("imputer",SimpleImputer(strategy="most_frequent")),("encoder",OneHotEncoder(handle_unknown="ignore",sparse_output=False))])

In [8]:
preprocessor = ColumnTransformer(transformers=[("num",numerical_pipeline,numerical_features),("cat",categorical_pipeline,categorical_features)])

In [9]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape :", X_test_processed.shape)

Processed X_train shape: (5634, 2828)
Processed X_test shape : (1409, 2828)


In [10]:
print("Original feature count:",
    X_train.shape[1])

print("Processed feature count:",
    X_train_processed.shape[1])

Original feature count: 25
Processed feature count: 2828


In [11]:
feature_names = preprocessor.get_feature_names_out()

print("Total transformed features:", len(feature_names))
print("\nFirst 30 features:")
print(feature_names[:30])

Total transformed features: 2828

First 30 features:
['num__Zip Code' 'num__Latitude' 'num__Longitude' 'num__Tenure Months'
 'num__Monthly Charges' 'num__Total Charges' 'num__CLTV'
 'cat__City_Acampo' 'cat__City_Acton' 'cat__City_Adelanto'
 'cat__City_Adin' 'cat__City_Agoura Hills' 'cat__City_Aguanga'
 'cat__City_Ahwahnee' 'cat__City_Alameda' 'cat__City_Alamo'
 'cat__City_Albany' 'cat__City_Albion' 'cat__City_Alderpoint'
 'cat__City_Alhambra' 'cat__City_Aliso Viejo' 'cat__City_Alleghany'
 'cat__City_Alpaugh' 'cat__City_Alpine' 'cat__City_Alta'
 'cat__City_Altadena' 'cat__City_Alturas' 'cat__City_Alviso'
 'cat__City_Amador City' 'cat__City_Amboy']


In [14]:
train_array = np.asarray(X_train_processed, dtype=float)
test_array = np.asarray(X_test_processed, dtype=float)
print("NaN values in X_train_processed:",np.isnan(train_array).sum())
print("NaN values in X_test_processed:",np.isnan(test_array).sum())

NaN values in X_train_processed: 0
NaN values in X_test_processed: 0
